# 01. Master Merge & Target Synthesis
Fuses NDVI, Meteorological, and Demographic data. Engineers weather anomalies, 
prevents data leakage via chronological splitting, and synthesizes the binary 
risk target (1 = Stress, 0 = Safe) using the Vegetation Condition Index (VCI) approach.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../../data_pipeline/data/processed")

# 1. Load the clean artifacts
ndvi = pd.read_csv(DATA_DIR / "NDVI_cleaned_FINAL.csv")
met  = pd.read_csv(DATA_DIR / "Metrological_Cleaned_Data_2_FINAL.csv")
demo = pd.read_csv(DATA_DIR / "Demographic_FINAL.csv")

# THE FIX: drop the integer year/month first so the 'month' label stays unique,
# then promote the YYYY-MM key to 'month' to match NDVI
met = met.drop(columns=["year", "month"])
met = met.rename(columns={"month_key": "month"})

# 2. Master Merge (NDVI + Weather)
df = pd.merge(ndvi, met, on=["county", "month"], how="inner")

# 3. Merge Demographics (static county features)
df = pd.merge(df, demo, on="county", how="left")

# Graceful fallback: fill the 3 missing counties with the national median
demo_cols = [c for c in demo.columns if c != "county"]
for col in demo_cols:
    df[col] = df[col].fillna(df[col].median())

print(f"✅ Master Matrix Shape: {df.shape}")
df.head(3)

✅ Master Matrix Shape: (3008, 22)


,county,month,ndvi_mean,month_num,month_sin,month_cos,ndvi_lag1,ndvi_lag3,ndvi_roll3_mean,ndvi_roll3_std,...,temp_mean_c,rainfall_mm,soil_moisture,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation
0,Baringo,2020-04,0.608943,4,8.660254e-01,-0.500000,0.558089,0.608311,0.587549,0.026216,...,25.144306,373.7,0.222150,142518.0,100465.0,84426.0,80426.0,511.0,841.0,7165.0
1,Baringo,2020-05,0.680116,5,5.000000e-01,-0.866025,0.608943,0.596246,0.587760,0.026468,...,24.795699,213.2,0.247187,142518.0,100465.0,84426.0,80426.0,511.0,841.0,7165.0
2,Baringo,2020-06,0.678639,6,1.224647e-16,-1.000000,0.680116,0.558089,0.615716,0.061295,...,23.482222,200.4,0.226953,142518.0,100465.0,84426.0,80426.0,511.0,841.0,7165.0


In [5]:
# Extract month number (1-12) from the 'YYYY-MM' string to calculate seasonal anomalies
df['month_num'] = df['month'].str.split('-').str[1].astype(int)

# 1. Rainfall Anomaly: How does this month's rain compare to the historical average 
# for this specific county and this specific month?
hist_rain = df.groupby(['county', 'month_num'])['rainfall_mm'].transform('mean')
df['rainfall_anomaly'] = (df['rainfall_mm'] - hist_rain) / (hist_rain + 1e-6)

# 2. Soil Moisture Deficit: How far below "field capacity" (approx 0.45) is the soil?
df['soil_moisture_deficit'] = np.maximum(0, 0.45 - df['soil_moisture']) * 100

print("✅ Weather anomalies engineered.")

✅ Weather anomalies engineered.


In [6]:
# 1. Chronological Split (Prevents Data Leakage)
df['year'] = df['month'].str.split('-').str[0].astype(int)

train_mask = df['year'] <= 2023
test_mask  = df['year'] >= 2024

train_df = df[train_mask].copy()
test_df  = df[test_mask].copy()

# 2. Target Synthesis (Vegetation Condition Index Approach)
# Calculate the 25th percentile of NDVI STRICTLY on the training set.
# This represents the "historical baseline" of stress for each county.
thresholds = train_df.groupby('county')['ndvi_mean'].quantile(0.25).reset_index()
thresholds.rename(columns={'ndvi_mean': 'ndvi_threshold'}, inplace=True)

# Apply the threshold to both sets
train_df = train_df.merge(thresholds, on='county')
test_df  = test_df.merge(thresholds, on='county')

# Create the binary target: 1 = High Risk (NDVI drops below baseline), 0 = Safe
train_df['target_risk'] = (train_df['ndvi_mean'] < train_df['ndvi_threshold']).astype(int)
test_df['target_risk']  = (test_df['ndvi_mean']  < test_df['ndvi_threshold']).astype(int)

print(f"Train Risk Ratio: {train_df['target_risk'].mean():.2%}")
print(f"Test Risk Ratio:  {test_df['target_risk'].mean():.2%}")

Train Risk Ratio: 24.44%
Test Risk Ratio:  7.84%


In [8]:
FEATURE_COLS = [
    'month_sin', 'month_cos',             # Seasonality
    'ndvi_lag1', 'ndvi_lag3',             # Memory
    'ndvi_roll3_mean', 'ndvi_roll3_std',  # Trends
    'ndvi_anomaly', 'ndvi_mom_change',    # Momentum
    'temp_mean_c', 'rainfall_mm', 'soil_moisture',          # Raw weather
    'rainfall_anomaly', 'soil_moisture_deficit',            # Engineered weather
    'total', 'farming', 'crop_production', 'livestock_production', 'irrigation'  # Demographics
]

# Safety check: report any feature that isn't in the frame before extracting
missing = [c for c in FEATURE_COLS if c not in train_df.columns]
if missing:
    raise KeyError(f"Features not in matrix: {missing}")

# Extract X and y
X_train, y_train = train_df[FEATURE_COLS], train_df['target_risk']
X_test,  y_test  = test_df[FEATURE_COLS],  test_df['target_risk']

# Save the final matrices to disk
train_df.to_csv(DATA_DIR / "train_matrix.csv", index=False)
test_df.to_csv(DATA_DIR / "test_matrix.csv", index=False)

print(f"\n💾 Saved train_matrix.csv: {X_train.shape}")
print(f"💾 Saved test_matrix.csv:  {X_test.shape}")
print(f"\nTrain risk ratio: {y_train.mean():.2%} | Test risk ratio: {y_test.mean():.2%}")


💾 Saved train_matrix.csv: (2115, 18)
💾 Saved test_matrix.csv:  (893, 18)

Train risk ratio: 24.44% | Test risk ratio: 7.84%
